# Hung #39 — Kaggle Development verification (diagnostic)

Attach `hung-39-f523718.bundle` as a **private Kaggle Input** before running. Optionally attach `inputs-urbanev-SZH-v1.zip`; otherwise the notebook downloads that public archive. Use **GPU T4 x2** and Internet on. Code and input are checked by SHA-256 before the 7/14/28-day, 7/11/19 verification matrix. Save a version after execution to retain `/kaggle/working` outputs. Kaggle acceptance by the team is still pending; no Final/test data are used for selection.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

CODE_SHA = "f52371823c1ddbf765eacfe2b89f7bcc0baf4869"
CODE_BUNDLE_SHA256 = "e29f1526ae1e01dacaebe46dd947c6a1d2583749300f0894b2e204c7b5f8002b"
INPUT_SHA256 = "2b8d1a47edebe066543f8f0aec5897c23945edcef62ba10357b1cb95e5ba27c7"
RUN_ID = "hung-39-kaggle-verification-20261025-v2"
REPO = Path("/kaggle/working/hung-sprint2-code")
OUTPUT_ROOT = Path("/kaggle/working") / RUN_ID
INPUT_ROOT = Path("/kaggle/temp/hung-39-input")
ARCHIVE = Path("/kaggle/temp/inputs-urbanev-SZH-v1.zip")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


bundle_locations = sorted(Path("/kaggle/input").rglob("hung-39-f523718.bundle"))
bundle_locations += [
    path for path in [Path("/kaggle/working/hung-39-f523718.bundle")] if path.is_file()
]
if len(bundle_locations) != 1:
    raise FileNotFoundError(
        "Attach exactly one Kaggle Input containing hung-39-f523718.bundle; "
        f"found {len(bundle_locations)}. Do not expose repository credentials."
    )
code_bundle = bundle_locations[0]
if sha256(code_bundle) != CODE_BUNDLE_SHA256:
    raise ValueError("Pinned code bundle SHA-256 mismatch: " + str(code_bundle))
if REPO.exists():
    check = subprocess.run(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True
    )
    if check.returncode or check.stdout.strip() != CODE_SHA:
        raise ValueError("Existing code checkout differs; use a fresh Kaggle session: " + str(REPO))
else:
    subprocess.run(["git", "clone", str(code_bundle), str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_SHA], check=True)
subprocess.run(["git", "-C", str(REPO), "bundle", "verify", str(code_bundle)], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_SHA, actual
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-colab.txt")], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "-r",
        str(REPO / "requirements-cuda.txt"),
    ],
    check=True,
)
probe = subprocess.check_output(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')",
    ],
    text=True,
).strip()
if not probe.startswith("2.8.0+cu126 True"):
    raise RuntimeError("Pinned CUDA probe failed: " + probe)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print(
    "code",
    actual,
    "bundle",
    CODE_BUNDLE_SHA256,
    "device probe",
    probe,
    "free_disk_gb",
    round(shutil.disk_usage("/kaggle/working").free / 1e9, 2),
)

In [ ]:
import gdown

input_archives = sorted(Path("/kaggle/input").rglob("inputs-urbanev-SZH-v1.zip"))
if len(input_archives) > 1:
    raise ValueError("Attach only one DATA-v1 input ZIP to Kaggle")
if input_archives:
    archive = input_archives[0]
else:
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.is_file():
        gdown.download(id="17pfYqnnSq0Uao2zDNdb8h7GAYfotu_w4", output=str(ARCHIVE), quiet=False)
    archive = ARCHIVE
if not archive.is_file() or archive.stat().st_size != 151254390 or sha256(archive) != INPUT_SHA256:
    raise ValueError("DATA-v1 archive size or SHA-256 mismatch: " + str(archive))
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.hung_sprint2.cli",
        "verify-input-zip",
        "--archive",
        str(archive),
        "--package",
        "inputs-urbanev-SZH-v1.zip",
    ],
    cwd=REPO,
    check=True,
)
if INPUT_ROOT.exists():
    raise FileExistsError(
        "Input extraction already exists; inspect this session before rerunning: " + str(INPUT_ROOT)
    )
INPUT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(archive) as source_zip:
    for member in source_zip.infolist():
        target = (INPUT_ROOT / member.filename).resolve()
        if not target.is_relative_to(INPUT_ROOT.resolve()) or target.exists():
            raise ValueError("Unsafe or duplicate ZIP member")
    source_zip.extractall(INPUT_ROOT)
INPUT_BASE = INPUT_ROOT / "data/inputs/issue18-input-handoff-v1-20261019"
assert INPUT_BASE.is_dir()
print("Verified DATA-v1", INPUT_SHA256, "archive", archive, "input base", INPUT_BASE)

In [ ]:
command = [
    sys.executable,
    "-m",
    "experiments.hung_sprint2.run_development",
    "--input-base",
    str(INPUT_BASE),
    "--selection",
    str(REPO / "docs/reports/hung-development-selection-20261025-v2.json"),
    "--config",
    str(REPO / "experiments/hung_sprint2/development_config_v1.json"),
    "--output-root",
    str(OUTPUT_ROOT),
    "--phase",
    "verification",
    "--device",
    "cuda",
    "--budgets",
    "7",
    "14",
    "28",
    "--seeds",
    "7",
    "11",
    "19",
    "--execute",
]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT_ROOT / "execution.log"
with log_path.open("a", encoding="utf-8") as log:
    log.write("COMMAND " + " ".join(command) + "\n")
    log.flush()
    process = subprocess.Popen(
        command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f"Development matrix failed with exit code {exit_code}; inspect {log_path}")
print("Matrix completed:", OUTPUT_ROOT)

In [ ]:
from collections import Counter

identity = json.loads((OUTPUT_ROOT / "matrix_identity.json").read_text())
assert identity["code"]["commit"] == CODE_SHA
assert identity["device"] == "cuda" and identity["phase"] == "verification"
assert identity["budgets"] == [7, 14, 28] and identity["seeds"] == [7, 11, 19]
manifests = sorted(OUTPUT_ROOT.rglob("run_manifest.json"))
if len(manifests) != 63:
    raise RuntimeError(f"Expected 63 completed stages, found {len(manifests)}")
summary = {
    "schema": "hung-39-kaggle-verification-summary-v1",
    "run_id": RUN_ID,
    "platform": "Kaggle",
    "code_sha": CODE_SHA,
    "input_zip_sha256": INPUT_SHA256,
    "matrix_identity_sha256": sha256(OUTPUT_ROOT / "matrix_identity.json"),
    "stage_count": len(manifests),
    "hashes_checked": 0,
    "stages": Counter(),
    "replay_90": [],
}
for path in manifests:
    row = json.loads(path.read_text())
    if row.get("status") != "completed":
        raise RuntimeError("Incomplete stage: " + str(path))
    rel = path.relative_to(OUTPUT_ROOT)
    stage = path.parent.name
    summary["stages"][stage] += 1
    checkpoint = path.parent / (
        "runner_checkpoint.pt"
        if stage.startswith("replay")
        else ("source_checkpoint.pt" if stage == "source" else "target_checkpoint.pt")
    )
    expected = (
        row.get("final_runner_checkpoint_sha256")
        if stage.startswith("replay")
        else row.get("checkpoint_sha256")
    )
    if not checkpoint.is_file() or sha256(checkpoint) != expected:
        raise RuntimeError("Checkpoint hash mismatch: " + str(checkpoint))
    summary["hashes_checked"] += 1
    if stage.startswith("replay"):
        journal = path.parent / "events.jsonl"
        expected_journal = row.get("event_journal", {}).get("sha256")
        if not journal.is_file() or sha256(journal) != expected_journal:
            raise RuntimeError("Event journal hash mismatch: " + str(journal))
        summary["hashes_checked"] += 1
    if stage == "replay-90":
        summary["replay_90"].append(
            {
                "path": str(rel),
                "metrics_kwh": row.get("metrics_kwh"),
                "coverage": row.get("coverage"),
                "device": row.get("device"),
            }
        )
summary["stages"] = dict(summary["stages"])
summary_path = OUTPUT_ROOT / "kaggle_verification_summary.json"
with summary_path.open("x", encoding="utf-8") as stream:
    json.dump(summary, stream, indent=2, sort_keys=True)
    stream.write("\n")
print(
    "verified",
    summary["stage_count"],
    "stages and",
    summary["hashes_checked"],
    "checkpoint/journal hashes",
)
print("summary", summary_path)